# Analyse du Programme Social du Gouvernement (PSGouv 2)
## Réalisations 2023 – juin 2024 · Répartition territoriale et sectorielle

**Auteur** : FRANCK HUBERSON KANGAH
**Date** : 17 Novembre 2026
**Source des données** : [data.gouv.ci](https://data.gouv.ci) — dataset officiel « Réalisations du PSGouv 2 de 2023 à juin 2024 »


### Objectif

Le PSGouv 2 est le programme social du gouvernement ivoirien. Ce notebook analyse
la répartition de ses réalisations entre les **33 régions et districts** du pays
et les **6 grands secteurs** d'intervention (Éducative, Sanitaire, Agricole,
Électrique et Hydraulique, Sociale, Professionnelle).

**Question centrale** : *Comment les investissements sociaux du PSGouv 2 se
répartissent-ils entre les régions et les secteurs, et quelles disparités
territoriales peut-on identifier ?*

### Démarche

1. **Inspection** du dataset brut (structure, typage, valeurs manquantes)
2. **Diagnostic qualité** : détection de valeurs aberrantes et d'incohérences
3. **Nettoyage documenté** : neutralisation des valeurs corrompues, conservation de 97 % des données
4. **Analyse exploratoire** : couverture territoriale, top/flop régional, focus Agnéby-Tiassa
5. **Conclusion** : insights clés et limites méthodologiques

### Table des matières

1. [Chargement des données](#1-chargement-des-données)
2. [Inspection initiale](#2-inspection-initiale)
3. [Qualité des données](#3-qualité-des-données)
4. [Détection des valeurs aberrantes](#4-détection-des-valeurs-aberrantes)
5. [Nettoyage et validation](#5-nettoyage-et-validation)
6. [Analyse exploratoire](#6-analyse-exploratoire)
   - 6.1 [Couverture territoriale par secteur](#61-couverture-territoriale-par-secteur)
   - 6.2 [Top/flop régional sur 3 indicateurs phares](#62-topflop-régional-sur-3-indicateurs-phares)
   - 6.3 [Profil de l'Agnéby-Tiassa](#63-profil-de-lagnéby-tiassa)
7. [Conclusion](#7-conclusion)

---

## 1. Chargement des données

Le fichier est un CSV exporté depuis le portail national des données ouvertes.
On le charge avec pandas et on vérifie que la lecture s'est bien passée.

In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("Psgouv 2023- juin 2024 final.csv")
df.head()

,District,Région,types d'initative,Catégorie,Désignation,Résultats
0,ABIDJAN,ABIDJAN,Éducative,Infrastructure éducative,Collèges construits (Nombre),2
1,LAGUNES,AGNEBY-TIASSA,Éducative,Infrastructure éducative,Collèges construits (Nombre),3
2,WOROBA,BAFING,Éducative,Infrastructure éducative,Collèges construits (Nombre),0
3,SAVANES,BAGOUE,Éducative,Infrastructure éducative,Collèges construits (Nombre),6
4,LACS,BELIER,Éducative,Infrastructure éducative,Collèges construits (Nombre),0


---

## 2. Inspection initiale

Avant toute analyse, on caractérise la **forme** du dataset (combien de lignes,
de colonnes), le **typage** des colonnes, et les **modalités** des variables
catégorielles. On vérifie aussi l'absence de doublons.

In [27]:
df.shape

(1337, 6)

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   District           1337 non-null   str  
 1   Région             1337 non-null   str  
 2   types d'initative  1337 non-null   str  
 3   Catégorie          1337 non-null   str  
 4   Désignation        1337 non-null   str  
 5   Résultats          1337 non-null   int64
dtypes: int64(1), str(5)
memory usage: 62.8 KB


**Observations** :

- Le dataset contient **1 337 lignes** et **6 colonnes**.
- **Aucune valeur manquante** sur les 6 colonnes (1 337 non-null partout).
- La colonne `Résultats` est bien en `int64` (numérique entier).
- Les 5 autres colonnes sont des chaînes de caractères (dimensions).

Le dataset est en **format long** : une ligne = un couple (région × indicateur).

In [29]:
df["types d'initative"].value_counts()

types d'initative
Sanitaire                    528
Agricole                     396
Éducative                    231
Électrique et Hydraulique    132
Sociale                       33
Professionnelle               17
Name: count, dtype: int64

In [30]:
df["Catégorie"].value_counts()

Catégorie
Manioc                                    330
Matériel éducatif                         165
Vaccin                                    132
Pompe à mobilité humaine (PMH)             99
Etablissement sanitaire et hospitalier     99
Accouchement                               99
Lutte contre le paludisme                  99
Infrastructure éducative                   66
Produit sanguin                            66
Agriculture maraichère                     66
Electrification                            33
Sociale                                    33
Conditions de vie des ménages              33
Professionnelle                            17
Name: count, dtype: int64

In [32]:
df["Région"].nunique()
print(sorted(df["Région"].unique()))

['ABIDJAN', 'AGNEBY-TIASSA', 'BAFING', 'BAGOUE', 'BELIER', 'BERE', 'BOUNKANI', 'CAVALLY', 'FOLON', 'GBEKE', 'GBOKLE', 'GOH', 'GONTOUGO', 'GRANDS-PONTS', 'GUEMON', 'HAMBOL', 'HAUT-SASSANDRA', 'IFFOU', 'INDENIE-DJUABLIN', 'KABADOUGOU', 'LA ME', 'LOH-DJIBOUA', 'MARAHOUE', 'MORONOU', "N'ZI", 'NAWA', 'PORO', 'SAN-PEDRO', 'SUD-COMOE', 'TCHOLOGO', 'TONKPI', 'WORODOUGOU', 'YAMOUSSOUKRO']


**Observations** :

- **6 secteurs** d'intervention, dominés par le Sanitaire (528 lignes, 39 %)
  et l'Agricole (396 lignes, 30 %). Les secteurs Sociale (33) et
  Professionnelle (17) sont marginaux.
- **14 catégories** distinctes. « Manioc » (330 lignes) est la catégorie la
  plus fréquente, reflet du programme agricole centré sur cette filière.
- **33 régions/districts** exactement, avec des noms propres (pas de doublon
  d'orthographe) : 31 régions + Abidjan + Yamoussoukro.

In [33]:
df.duplicated().sum()

np.int64(0)

**Observation** : aucun doublon strict. Chaque (région × indicateur) apparaît
une seule fois. Le dataset est structurellement sain.

## 3. Qualité des données

On vérifie maintenant la **distribution de `Résultats`**. Un mélange d'unités
(collèges, personnes vaccinées, montants en FCFA) se traduirait par un écart
anormal entre médiane et moyenne.

In [34]:
df["Résultats"].describe()

count    1.337000e+03
mean     4.064682e+06
std      8.334599e+07
min      0.000000e+00
25%      0.000000e+00
50%      6.000000e+01
75%      1.538300e+04
max      3.014466e+09
Name: Résultats, dtype: float64

**Observation critique** :

- **Médiane = 60** mais **moyenne = 4 064 682** : écart d'un facteur ~68 000.
- **Maximum = 3 milliards** (3 014 466 070).
- 75e percentile = 15 383, alors que le maximum est à 3 milliards.

Cet écart signe un **mélange d'unités dans la même colonne** : certaines
lignes comptent des bâtiments (2 collèges), d'autres des personnes
(297 462 enfants vaccinés), d'autres des montants en FCFA (3 milliards).
**Toute agrégation naïve de `Résultats` serait donc fausse.**

In [36]:
df.groupby("Désignation")["Résultats"].agg(["count", "min", "max", "median"])

,count,min,max,median
Désignation,,,,
Centres de santé CMU fonctionnel (nombre),33,10,84617,84.0
Collèges construits (Nombre),33,0,8,0.0
ESPC construits (nombre),33,0,22,1.0
ESPC réhabilités et équipés (nombre),33,0,20503,4.0
Enfants vaccinés HPV1 (nombre),33,4416,66638,13692.0
Enfants vaccinés au Penta 3 (Nombre),33,7803,297462,32491.0
Enfants vaccinés rougeole/Rubéole (Nombre),33,7472,281360,37664.0
Estimation de vente,33,0,3014466070,0.0
Femmes enceintes vaccinées tétanos (Nombre),33,0,275304,18239.0


**Observations** :

- **41 indicateurs** au total, dont **40 présents dans les 33 régions**.
  L'indicateur « Nombre de jeunes mis en stage » n'existe que dans **17 régions**
  (programme pilote).
- **3 indicateurs présentent des maxima physiquement impossibles** :

| Indicateur | Max observé | Max plausible | Verdict |
|---|---:|---:|---|
| Centres de santé CMU fonctionnel | 84 617 | ~200 | ❌ impossible |
| ESPC réhabilités et équipés | 20 503 | ~300 | ❌ impossible |
| Césarienne (interventions) | 841 415 | < accouchements | ❌ impossible |

- **1 indicateur est très suspect** : Manuels scolaires (max 5 297 658 pour une
  seule région).

Ces valeurs ne peuvent pas appartenir à l'indicateur sur la ligne où elles se
trouvent. Il faut enquêter avant de nettoyer.

## 4. Détection des valeurs aberrantes

On combine deux approches complémentaires :

- **Approche statistique** : une valeur est suspecte si elle dépasse **10 fois
  la médiane non-nulle** de son indicateur (la médiane non-nulle évite que les
  indicateurs « zero-inflated » comme le manioc ne faussent la référence).
- **Approche métier** : certaines incohérences sont **logiquement impossibles**
  indépendamment des autres régions (ex. césarienne > accouchements).

In [37]:
# Médiane non-nulle par indicateur (référence)
def median_nonzero(s):
    nz = s[s > 0]
    return nz.median() if len(nz) > 0 else 0

ref = df.groupby("Désignation")["Résultats"].transform(median_nonzero)
df["ratio_median"] = df["Résultats"] / ref.replace(0, np.nan)
df["flag_outlier"] = df["ratio_median"] > 10

print(f"Valeurs flaggées : {df['flag_outlier'].sum()} / {len(df)} "
      f"({df['flag_outlier'].mean()*100:.1f}%)")

print("\n--- Détail par indicateur ---")
print(df[df["flag_outlier"]].groupby("Désignation").size()
      .sort_values(ascending=False).to_string())

Valeurs flaggées : 40 / 1337 (3.0%)

--- Détail par indicateur ---
Désignation
Centres de santé CMU fonctionnel (nombre)                                                                                                               12
Nombre d'interventions chirurgicales, Césarienne (total)                                                                                                12
Nombre Total de cas de paludisme confirmés et traités chez la femme enceinte                                                                             3
ESPC réhabilités et équipés (nombre)                                                                                                                     2
SIG - Nombre d’unités de poche de sang reçus à l'hôpital de référence - CHR+HG (Sang total et Concentré Erythrocytaire Adulte + Pédiatrique)             2
Estimation de vente                                                                                                                               

**Observations** :

- **40 valeurs flaggées sur 1 337 (3,0 %)** : le taux de corruption est faible
  et gérable.
- **Deux indicateurs concentrent 24 des 40 flags (60 %)** : Centres de santé
  CMU (12) et Césarienne (12). Les **mêmes 12 régions** reviennent.
- Le motif n'est pas aléatoire : c'est la signature d'un **bug d'export**
  systématique, pas d'erreurs de saisie indépendantes.

In [38]:
regions_cmu = set(df[(df["Désignation"]=="Centres de santé CMU fonctionnel (nombre)")
                     & df["flag_outlier"]]["Région"])
regions_cesa = set(df[(df["Désignation"]=="Nombre d'interventions chirurgicales, Césarienne (total)")
                      & df["flag_outlier"]]["Région"])

print("Les 12 régions sont identiques ?", regions_cmu == regions_cesa)
print("Uniquement CMU :", regions_cmu - regions_cesa)
print("Uniquement Césa :", regions_cesa - regions_cmu)

Les 12 régions sont identiques ? True
Uniquement CMU : set()
Uniquement Césa : set()


**Confirmation** : les 12 régions sont **rigoureusement identiques** sur les
deux indicateurs (et sur un troisième, le paludisme chez la femme enceinte,
détecté plus loin). Ce n'est pas une coïncidence : un **bug d'export ou de
fusion de tables** a corrompu les données de ces 12 régions sur 3 indicateurs
précis. Les 36 autres indicateurs de ces régions sont exploitables.

In [39]:
wide = df.pivot_table(index="Région", columns="Désignation",
                      values="Résultats", aggfunc="first")

CESA   = "Nombre d'interventions chirurgicales, Césarienne (total)"
ACC    = "Nombre d'Accouchement dans l’établissement (total)"
PALU   = "Nombre Total de cas de paludisme confirmés et traités"
PALU_F = "Nombre Total de cas de paludisme confirmés et traités chez la femme enceinte"
PALU_E = "Nombre Total de cas de paludisme confirmés et traités chez les enfants de 0-4 ans"
SANG_R = "SIG - Nombre d’unités de poche de sang reçus à l'hôpital de référence - CHR+HG (Sang total et Concentré Erythrocytaire Adulte + Pédiatrique)"
SANG_T = "SIG - Nombre d’unités de poche de sang transfusées à l'hôpital de référence - CHR + HG (Sang total et Concentré Erythrocytaire Adulte + Pédiatrique)"

regles = {
    "Césarienne > Accouchements"   : (CESA,   ACC),
    "Palu femme enceinte > total"  : (PALU_F, PALU),
    "Palu enfants 0-4 ans > total" : (PALU_E, PALU),
    "Sang transfusé > sang reçu"   : (SANG_T, SANG_R),
}

for nom, (a, b) in regles.items():
    mask = wide[a] > wide[b]
    print(f"\n{nom} — {mask.sum()} région(s)")
    if mask.sum() > 0:
        print(wide[mask][[a, b]].to_string())


Césarienne > Accouchements — 12 région(s)
Désignation   Nombre d'interventions chirurgicales, Césarienne (total)  Nombre d'Accouchement dans l’établissement (total)
Région                                                                                                                    
LA ME                                                         215601.0                                              2636.0
LOH-DJIBOUA                                                   347068.0                                              2235.0
MARAHOUE                                                      383035.0                                              1295.0
MORONOU                                                       254639.0                                               655.0
N'ZI                                                          143229.0                                               973.0
NAWA                                                          841415.0                          

**Observations** :

- **Césarienne > Accouchements** : 12 régions. Les mêmes 12 que pour le CMU.
  Exemple : NAWA affiche 841 415 césariennes pour 4 456 accouchements — une
  césarienne étant un accouchement, c'est **logiquement impossible**.
- **Paludisme femme enceinte > paludisme total** : 12 régions. Même liste.
- **Paludisme enfants 0-4 ans > total** : 2 régions (SUD-COMOE, YAMOUSSOUKRO).
- **Sang transfusé > sang reçu** : 5 régions, mais avec des écarts < 1 %
  (ex. AGNEBY-TIASSA : 9 092 transfusées vs 9 034 reçues). Ce sont des
  **faux positifs** : bruit d'inventaire, pas corruption. On les conserve.

**Décision** : neutraliser les 3 indicateurs corrompus pour les 12 régions
(36 valeurs) + l'incohérence palu enfants (2 valeurs) = **38 valeurs → NaN**.

## 5. Nettoyage et validation

On crée une colonne `Résultats_validés` qui neutralise les valeurs corrompues
(`NaN`) **sans supprimer l'original** (traçabilité). On conserve aussi deux
colonnes de flags (`flag_corrompu`, `flag_douteux`) pour documenter chaque
décision.

In [43]:
# 1. Régions et indicateurs corrompus (bug d'export confirmé)
REGIONS_CORROMPUES = {
    "LA ME", "LOH-DJIBOUA", "MARAHOUE", "MORONOU", "N'ZI", "NAWA",
    "PORO", "SAN-PEDRO", "SUD-COMOE", "TCHOLOGO", "TONKPI", "YAMOUSSOUKRO"
}
INDICATEURS_CORROMPUS = {
    "Centres de santé CMU fonctionnel (nombre)",
    "Nombre d'interventions chirurgicales, Césarienne (total)",
    "Nombre Total de cas de paludisme confirmés et traités chez la femme enceinte",
}
mask_corrompu = (
    df["Région"].isin(REGIONS_CORROMPUES)
    & df["Désignation"].isin(INDICATEURS_CORROMPUS)
)

# 2. Incohérence palu enfants > total
regions_palu_enfant = wide[wide[PALU_E] > wide[PALU]].index.tolist()
mask_palu_enfant = (
    df["Région"].isin(regions_palu_enfant) & (df["Désignation"] == PALU_E)
)

# 3. Cas douteux (gardés mais flaggés)
mask_douteux = (
    ((df["Désignation"] == "Manuels scolaires (Nombre)") & (df["Région"] == "FOLON"))
    | ((df["Désignation"] == SANG_T) & (df["Région"] == "GOH"))
)

# 4. Application
df["flag_corrompu"] = mask_corrompu | mask_palu_enfant
df["flag_douteux"]  = mask_douteux
df["Résultats_validés"] = df["Résultats"].where(~df["flag_corrompu"])

# 5. Contrôles
print(f"Valeurs invalidées (NaN)      : {df['flag_corrompu'].sum()}")
print(f"Valeurs douteuses (flaggées)  : {df['flag_douteux'].sum()}")
print(f"Valeurs validées              : {df['Résultats_validés'].notna().sum()} / {len(df)}")
print(f"Taux de conservation          : {df['Résultats_validés'].notna().mean()*100:.1f}%")

# 6. Vérification : plus aucune incohérence métier
wide_clean = df.pivot_table(index="Région", columns="Désignation",
                            values="Résultats_validés", aggfunc="first")
print("\nCésarienne > Accouchements restants :",
      (wide_clean[CESA] > wide_clean[ACC]).sum())
print("Palu femme > total restants        :",
      (wide_clean[PALU] < wide_clean[PALU_F]).sum())

Valeurs invalidées (NaN)      : 38
Valeurs douteuses (flaggées)  : 2
Valeurs validées              : 1299 / 1337
Taux de conservation          : 97.2%

Césarienne > Accouchements restants : 0
Palu femme > total restants        : 0


**Résultat du nettoyage** :

- **38 valeurs invalidées** (36 corrompues + 2 incohérentes)
- **2 valeurs douteuses** conservées mais flaggées (FOLON manuels, GOH sang)
- **1 299 valeurs validées sur 1 337** → **taux de conservation : 97,2 %**
- **0 incohérence métier restante** (césarienne ≤ accouchements partout,
  palu femme ≤ total partout)

Le dataset est désormais **propre, auditable et défendable**. Chaque décision
est documentée par une colonne de flag.

## 6. Analyse exploratoire

### 6.1 Couverture territoriale par secteur

**Question** : combien d'indicateurs de chaque secteur ont une valeur > 0
dans chaque région ? On mesure la **couverture**, pas le volume — additionner
des collèges et des montants FCFA n'aurait aucun sens.

In [44]:
secteur_map = df.drop_duplicates("Désignation").set_index("Désignation")["types d'initative"]
coverage = (wide_clean > 0).T.groupby(secteur_map).sum().T
total_indicateurs = secteur_map.value_counts()
coverage_pct = (coverage / total_indicateurs * 100).round(1)

print("Couverture par secteur (% d'indicateurs avec valeur > 0) :")
print(coverage_pct.sort_values("Sanitaire", ascending=False).to_string())

Couverture par secteur (% d'indicateurs avec valeur > 0) :
types d'initative  Agricole  Professionnelle  Sanitaire  Sociale  Éducative  Électrique et Hydraulique
Région                                                                                                
AGNEBY-TIASSA          66.7            100.0      100.0    100.0       71.4                       75.0
GRANDS-PONTS            0.0              0.0      100.0    100.0       28.6                       75.0
BERE                    0.0              0.0      100.0    100.0       57.1                      100.0
CAVALLY                 0.0              0.0      100.0    100.0       57.1                       75.0
GBOKLE                  0.0              0.0      100.0    100.0       42.9                      100.0
WORODOUGOU              0.0              0.0      100.0    100.0       57.1                      100.0
HAUT-SASSANDRA          0.0            100.0      100.0    100.0       28.6                       75.0
GUEMON        

In [45]:
angles_morts = []
for region in coverage_pct.index:
    for secteur in coverage_pct.columns:
        if coverage_pct.loc[region, secteur] == 0:
            angles_morts.append({"Région": region, "Secteur": secteur})
angles_morts_df = pd.DataFrame(angles_morts)
print(f"Nombre total d'angles morts : {len(angles_morts_df)}")
print("\nPar secteur :")
print(angles_morts_df["Secteur"].value_counts().to_string())

Nombre total d'angles morts : 44

Par secteur :
Secteur
Agricole           25
Professionnelle    18
Éducative           1


**Observations** :

- **Sanitaire** : seul secteur universel (68,8 % à 100 % de couverture dans
  toutes les régions). C'est le socle du programme.
- **Agricole** : **25 régions sur 33 sans aucun indicateur agricole > 0**.
  Le programme est concentré sur 8 régions (AGNEBY-TIASSA, GBEKE, GONTOUGO,
  BELIER, HAMBOL, TCHOLOGO, YAMOUSSOUKRO, PORO) — logique de bassins de
  production.
- **Professionnelle** : 18 régions sans « jeunes mis en stage » — mais
  l'indicateur n'existe que dans 17 régions au total. C'est un **programme
  pilote**, pas un angle mort.
- **SAN-PEDRO** : seule région avec **0 % en éducative** (7 indicateurs tous
  à 0). Anomalie à investiguer (grande région portuaire, absence totale peu
  crédible).

**Limite méthodologique** : un 0 % peut signifier soit l'absence du programme,
soit un résultat nul (ex. récolte nulle). Le dataset ne permet pas de
distinguer les deux.

### 6.2 Top/flop régional sur 3 indicateurs phares

On choisit 3 indicateurs bien renseignés (33 régions), un par secteur majeur,
en excluant les indicateurs corrompus.

In [47]:
indicateurs_phares = {
    "Kits scolaires (Nombre d'éleve beneficiaire)": "Éducative",
    "Enfants vaccinés au Penta 3 (Nombre)": "Sanitaire",
    "Localités électrifiées (Nombre)": "Électrique et Hydraulique",
}

for indicateur, secteur in indicateurs_phares.items():
    serie = wide_clean[indicateur].dropna().sort_values(ascending=False)
    total = serie.sum()
    part = (serie / total * 100).round(1)
    print(f"\n{'='*60}\n{secteur} — {indicateur}")
    print(f"Total national : {total:,.0f}".replace(",", " "))
    print(f"{'='*60}")
    print("\nTOP 5 :")
    for r in serie.head(5).index:
        print(f"  {r:20s} {serie[r]:>10,.0f}  ({part[r]:>5.1f} %)".replace(",", " "))
    print("\nFLOP 5 :")
    for r in serie.tail(5).index:
        print(f"  {r:20s} {serie[r]:>10,.0f}  ({part[r]:>5.1f} %)".replace(",", " "))
    print(f"\nConcentration top 3 : {part.head(3).sum():.1f} % du total national")


Éducative — Kits scolaires (Nombre d'éleve beneficiaire)
Total national : 3 744 841

TOP 5 :
  ABIDJAN                 673 209  ( 18.0 %)
  HAUT-SASSANDRA          262 413  (  7.0 %)
  NAWA                    260 469  (  7.0 %)
  TONKPI                  198 251  (  5.3 %)
  LOH-DJIBOUA             155 311  (  4.1 %)

FLOP 5 :
  BOUNKANI                 41 514  (  1.1 %)
  KABADOUGOU               36 041  (  1.0 %)
  BAFING                   35 730  (  1.0 %)
  FOLON                    14 740  (  0.4 %)
  SAN-PEDRO                     0  (  0.0 %)

Concentration top 3 : 32.0 % du total national

Sanitaire — Enfants vaccinés au Penta 3 (Nombre)
Total national : 1 615 294

TOP 5 :
  ABIDJAN                 297 462  ( 18.4 %)
  HAUT-SASSANDRA          111 703  (  6.9 %)
  NAWA                    103 545  (  6.4 %)
  GBEKE                    80 077  (  5.0 %)
  TONKPI                   75 828  (  4.7 %)

FLOP 5 :
  YAMOUSSOUKRO             21 013  (  1.3 %)
  N'ZI                     16 31

**Observations** :

- **ABIDJAN domine l'éducatif (18,0 %) et le sanitaire (18,4 %)** — cohérent
  avec son poids démographique (~20 % de la population). Le programme est donc
  **proportionnellement distribué**, pas concentré sur la capitale
  (concentration top 3 modérée : ~32 %).
- **L'électrification est un programme rural** : BOUNKANI est 1er (17,4 %),
  PORO 2e (12,5 %), tandis qu'ABIDJAN est **31e** (0,2 %). Ce n'est pas un
  échec : Abidjan est déjà entièrement électrifiée, l'indicateur mesure un
  **besoin résiduel**, pas une performance.
- **Les 12 régions corrompues restent compétitives** sur les indicateurs non
  corrompus : NAWA est 3e sur les kits scolaires et 3e sur Penta 3. Cela valide
  la décision de ne neutraliser que les 3 indicateurs identifiés.

### 6.3 Profil de l'Agnéby-Tiassa

Focus sur une région spécifique : où se situe-t-elle dans le dispositif
national, et quelles sont ses spécificités ?

In [48]:
REGION = "AGNEBY-TIASSA"

rangs = wide_clean.rank(ascending=False, method="min")
synthese = pd.DataFrame({
    "Valeur":     wide_clean.loc[REGION],
    "Rang":       rangs.loc[REGION].astype(int),
    "Total_nat.": wide_clean.sum(),
})
synthese["Part_%"] = (synthese["Valeur"] / synthese["Total_nat."] * 100).round(2)
synthese = synthese.sort_values("Rang")

print(f"=== PROFIL {REGION} — {len(synthese)} indicateurs ===")
print(f"Rang médian : {int(synthese['Rang'].median())} / 33")
print(f"Top 5  : {(synthese['Rang'] <= 5).sum()}")
print(f"Top 10 : {(synthese['Rang'] <= 10).sum()}")
print(f"Flop 10: {(synthese['Rang'] >= 24).sum()}")

print("\n--- TOP 10 des meilleurs rangs ---")
print(synthese.head(10)[["Valeur","Rang","Part_%"]].to_string())
print("\n--- FLOP 5 des pires rangs ---")
print(synthese.tail(5)[["Valeur","Rang","Part_%"]].to_string())

=== PROFIL AGNEBY-TIASSA — 41 indicateurs ===
Rang médian : 6 / 33
Top 5  : 15
Top 10 : 30
Flop 10: 1

--- TOP 10 des meilleurs rangs ---
                                                                     Valeur  Rang  Part_%
Désignation                                                                              
Estimation de vente                                            3.014466e+09     1   88.84
Production en tonne de manioc                                  6.486400e+04     1   55.31
Transformation artisanale de manioc (attieke produit)          1.973900e+04     1   77.44
Transformation semi industrielle de manioc: manioc acheté      6.544000e+03     1   61.57
Transformation artisanale de manioc (placali produit produit)  6.680000e+02     1   68.30
Transformation semi industrielle de manioc: attieke produit    2.833000e+03     1   75.39
Transformation artisanale de manioc (manioc acheté)            4.826400e+04     1   69.38
Superficie (HA) pour la production de manioc        

**Observations** :

- **Rang médian : 6 / 33** — l'Agnéby-Tiassa est dans le **premier quart**
  des régions les mieux servies.
- **15 indicateurs dans le top 5**, **30 dans le top 10**, **1 seul dans le
  flop 10**.
- **Points forts** : la région domine la filière manioc (1re sur 7 indicateurs
  agricoles, 55 % à 88 % du total national) et se classe 2e sur les tables
  bancs distribués.
- **Points faibles** : l'électrification (27e, 0,3 % du national) et la
  maintenance des PMH (22e). Ce sont les deux axes où la région est en retrait.

## 7. Conclusion

### Résultats clés

1. **Le dataset est structurellement sain** (1 337 lignes, 0 doublon, 0 valeur
   manquante) mais **contient un bug d'export** : 36 valeurs corrompues sur
   3 indicateurs et 12 régions, confirmées par des règles métier impossibles.
2. **97,2 % des données sont conservées** après nettoyage documenté. Les
   décisions sont traçables (colonnes de flags, original préservé).
3. **Le Sanitaire est le seul secteur universel** (couverture 68,8 %–100 %).
   L'Agricole est concentré sur 8 régions, le Professionnel est un programme
   pilote (17 régions).
4. **La distribution est proportionnelle à la population** : Abidjan domine
   les volumes absolus, mais la concentration top 3 reste modérée (~32 %).
   L'électrification est un programme rural (BOUNKANI 1er, ABIDJAN 31e).
5. **L'Agnéby-Tiassa est dans le premier quart** (rang médian 6/33), portée
   par la filière manioc, en retrait sur l'électrification.

### Limites méthodologiques

- **Absence de données de population** : les volumes bruts ne peuvent pas être
  normalisés par habitant.
- **Un 0 % de couverture ne distingue pas** « programme absent » de
  « résultat nul ».
- **L'unité des montants FCFA n'est pas homogène** avec les comptages : aucune
  agrégation financière nationale n'a été tentée.
- **Le bug d'export n'a pas pu être corrigé** (valeurs neutralisées, pas
  reconstituées) : les 38 valeurs perdues représentent 2,8 % du dataset.

### Prochaines étapes

Construire un **dashboard interactif** (Power BI)